<a href="https://colab.research.google.com/github/ramciel/ramciel.github.io/blob/main/Programming_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import networkx
import scipy.linalg as lin

In [4]:
# Construct Huckel matrix for a linear polyene given that alpha = 0, beta = -1
def linear_polyene(n_atoms):
    huckel_matrix = np.zeros((n_atoms, n_atoms))
    for i in range(n_atoms):
        if i > 0:
            huckel_matrix[i, i - 1] = -1
        if i < n_atoms - 1:
            huckel_matrix[i, i + 1] = -1
    return huckel_matrix

# Construct Huckel matrix for a cyclic polyene ring
def cyclic_polyene(n_atoms):
    huckel_matrix = np.zeros((n_atoms, n_atoms))
    for i in range(n_atoms):
        if i > 0:
            huckel_matrix[i, i - 1] = -1
        if i < n_atoms - 1:
            huckel_matrix[i, i + 1] = -1
    # connect first and last atoms to close the ring
    huckel_matrix[0, n_atoms - 1] = huckel_matrix[n_atoms - 1, 0] = -1
    return huckel_matrix

# Build adjacency matrix for sp2-hybridized Platonic solids
def platonic_solid(n_vertices):
    if n_vertices == 4:
        graph = nx.tetrahedral_graph()
    elif n_vertices == 6:
        graph = nx.octahedral_graph()
    elif n_vertices == 8:
        graph = nx.cubical_graph()
    elif n_vertices == 12:
        graph = nx.icosahedral_graph()
    elif n_vertices == 20:
        graph = nx.dodecahedral_graph()
    huckel_matrix = nx.adjacency_matrix(graph).toarray() * -1
    return huckel_matrix

# Huckel matrix for naphthalene, using a standard 10‑carbon numbering
def naphthalene_matrix():
    # 10 p-orbitals, one on each carbon
    huckel_matrix = np.zeros((10, 10))

    huckel_matrix[0, 1] = huckel_matrix[1, 0] = -1
    huckel_matrix[1, 2] = huckel_matrix[2, 1] = -1
    huckel_matrix[2, 3] = huckel_matrix[3, 2] = -1
    huckel_matrix[3, 8] = huckel_matrix[8, 3] = -1
    huckel_matrix[4, 5] = huckel_matrix[5, 4] = -1
    huckel_matrix[5, 6] = huckel_matrix[6, 5] = -1
    huckel_matrix[6, 7] = huckel_matrix[7, 6] = -1
    huckel_matrix[7, 9] = huckel_matrix[9, 7] = -1
    huckel_matrix[8, 9] = huckel_matrix[9, 8] = -1
    huckel_matrix[8, 0] = huckel_matrix[0, 8] = -1
    huckel_matrix[9, 4] = huckel_matrix[4, 9] = -1

    return huckel_matrix

# Approximate Huckel matrix for Buckminsterfullerene (C60)
def c60_matrix():
    mat = np.zeros((60, 60))
    for i in range(59):
        mat[i, i + 1] = -1

    # connect additional bonds (based on numbering scheme)
    bond_pairs = [
        (0, 4), (0, 8), (1, 11), (2, 14), (3, 17), (5, 19), (6, 21),
        (7, 24), (9, 25), (10, 28), (12, 29), (13, 32), (15, 33),
        (16, 36), (18, 37), (20, 39), (22, 41), (23, 43), (26, 44),
        (27, 46), (30, 47), (31, 49), (34, 50), (35, 52),
        (38, 53), (40, 54), (42, 56), (45, 57), (48, 58),
        (51, 59), (55, 59)
    ]
    for (i, j) in bond_pairs:
        mat[i, j] = mat[j, i] = -1
    return mat

# Calculate eigenvalues (energies)
def calculate_energies(huckel_matrix):
    eigenvalues, _ = lin.eigh(huckel_matrix)
    return np.round(eigenvalues, 4)

# Count degeneracies of energy levels
def compute_degeneracy(eigenvalues):
    degeneracy_dict = {}
    for e in eigenvalues:
        degeneracy_dict[e] = degeneracy_dict.get(e, 0) + 1
    return degeneracy_dict

molecule_type = input("Enter molecule type (linear, cyclic, platonic, naphthalene, or buckminsterfullerene): ")

if molecule_type in ["linear", "cyclic", "platonic"]:
    n_atoms = int(input("Enter number of atoms: "))

if molecule_type == "linear":
    adj_matrix = linear_polyene(n_atoms)
elif molecule_type == "cyclic":
    adj_matrix = cyclic_polyene(n_atoms)
elif molecule_type == "platonic":
    adj_matrix = platonic_solid(n_atoms)
elif molecule_type == "naphthalene":
    adj_matrix = naphthalene_matrix()
elif molecule_type in ["buckminsterfullerene", "c60"]:
    adj_matrix = c60_matrix()
else:
    raise ValueError("Unknown molecule type entered.")

# Solve Huckel eigenvalue problem
e_vals = calculate_energies(adj_matrix)
degeneracy = compute_degeneracy(e_vals)

# Output results
print("\nπ-Energy levels:")
print(e_vals)
print("\nDegeneracies:")
for i, j in sorted(degeneracy.items()):
    print(f"{i}: {j}")

Enter molecule type (linear, cyclic, platonic, naphthalene, or buckminsterfullerene): naphthalene

π-Energy levels:
[-2.3028 -1.8608 -1.     -0.618  -0.618  -0.2541  1.3028  1.618   1.618
  2.1149]

Degeneracies:
-2.3028: 1
-1.8608: 1
-1.0: 1
-0.618: 2
-0.2541: 1
1.3028: 1
1.618: 2
2.1149: 1
